# 05 — Condition coverage by location

Quick manifest analysis for kept images only. Computes condition coverage from `manifest.csv` grouped by `location`, `time_of_day`, and `weather`, with train/val split views.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

BASE = '..'
MANIFEST_CSV = f'{BASE}/manifest.csv'

df = pd.read_csv(MANIFEST_CSV)
kept_df = df[df['status'].eq('kept')].copy()

for col in ['location', 'time_of_day', 'weather', 'split']:
    kept_df[col] = kept_df[col].astype(str).str.lower().str.strip()

print(f'Manifest rows: {len(df)}')
print(f'Kept rows: {len(kept_df)}')
kept_df.head()

In [ ]:
condition_counts = (
    kept_df
    .groupby(['location', 'time_of_day', 'weather', 'split'])
    .size()
    .reset_index(name='n_images')
    .sort_values(['location', 'time_of_day', 'weather', 'split'])
)

condition_counts.head(20)

In [ ]:
coverage_by_location = (
    kept_df
    .assign(condition=kept_df['time_of_day'] + ' / ' + kept_df['weather'])
    .groupby(['location', 'condition'])
    .size()
    .unstack(fill_value=0)
)

coverage_by_location['total'] = coverage_by_location.sum(axis=1)
coverage_by_location = coverage_by_location.sort_values('total', ascending=False)
coverage_plot_df = coverage_by_location.drop(columns='total')

split_by_location = (
    kept_df
    .groupby(['location', 'split'])
    .size()
    .unstack(fill_value=0)
    .reindex(coverage_by_location.index)
)

fig, axes = plt.subplots(2, 1, figsize=(16, 13), sharex=True)

coverage_plot_df.plot(
    kind='bar',
    stacked=True,
    width=0.85,
    colormap='tab20',
    ax=axes[0],
)
axes[0].set_title('Per-location condition coverage: kept images only')
axes[0].set_xlabel('')
axes[0].set_ylabel('Image count')
axes[0].legend(title='Time of day / weather', bbox_to_anchor=(1.02, 1), loc='upper left')

split_by_location.plot(
    kind='bar',
    stacked=True,
    width=0.85,
    color={'train': '#4c78a8', 'val': '#f58518'},
    ax=axes[1],
)
axes[1].set_title('Train/val split by location: kept images only')
axes[1].set_xlabel('Location')
axes[1].set_ylabel('Image count')
axes[1].legend(title='Split', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
split_by_location

In [ ]:
condition_split = (
    kept_df
    .assign(condition=kept_df['time_of_day'] + ' / ' + kept_df['weather'])
    .groupby(['condition', 'split'])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

ax = condition_split.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 5),
    width=0.8,
    color={'train': '#4c78a8', 'val': '#f58518'},
)
ax.set_title('Train/val split by condition: kept images only')
ax.set_xlabel('Time of day / weather')
ax.set_ylabel('Image count')
ax.legend(title='Split', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()